# Lab — Temperature & sampling
<a id="top"></a>

```yaml
title: Lab — Temperature & sampling
keywords: temperature, softmax, sampling, variance, top-p, top-k
estimated duration: 25
```

> **Lesson L01 lab.** Objective 5 — explain temperature and sampling.
> ⚠️ **Problem 2 is live** (needs `ANTHROPIC_API_KEY`); Problem 1 is offline.
> Fill each `# TODO`; check against the solutions notebook.

**Contents**

1. [Reshape a distribution with temperature (offline)](#1-reshape-a-distribution-with-temperature-offline)
2. [Measure spread at two temperatures (live)](#2-measure-spread-at-two-temperatures-live)
3. [Pick a temperature per task](#3-pick-a-temperature-per-task)
4. [Why temperature 0 still varies](#4-why-temperature-0-still-varies)

In [ ]:
from __future__ import annotations

import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

transformers.logging.set_verbosity_error()  # quiet advisory HF logs in committed output

TOKENIZER = GPT2TokenizerFast.from_pretrained("gpt2")
MODEL = GPT2LMHeadModel.from_pretrained("gpt2").eval()
inputs = TOKENIZER("Water is made of hydrogen and", return_tensors="pt")
with torch.no_grad():
    LOGITS = MODEL(**inputs).logits[0, -1]

## 1. Reshape a distribution with temperature (offline)

Implement `top_prob`: divide `LOGITS` by `temperature`, softmax, and return the probability of the single most-likely token. Call it at T=0.5 and T=2.0 — the sharp one should be higher.

In [ ]:
def top_prob(temperature: float) -> float:
    probs = torch.softmax(LOGITS / temperature, dim=-1)
    return float(probs.max())

In [ ]:
print(round(top_prob(0.5), 3), round(top_prob(2.0), 3))

## 2. Measure spread at two temperatures (live)

Implement `sample_n(temperature, n)` to `await client.achat(...)` on the coffee-shop prompt `n`
times and return the replies (`achat` is the async twin of `chat` — the course defaults to it;
see the K05 prework). Compare the number of *distinct* answers at T=0 vs T=1.

In [ ]:
from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message

client = AnthropicClient()
COFFEE = "Name a coffee shop on a rainy Seattle street. Just the name, one or two words."


async def sample_n(temperature: float, n: int) -> list[str]:
    replies = [
        await client.achat([Message.user(COFFEE)], max_tokens=16, temperature=temperature)
        for _ in range(n)
    ]
    return [reply.text.strip() for reply in replies]


print("T=0 distinct:", len(set(await sample_n(0.0, 5))))
print("T=1 distinct:", len(set(await sample_n(1.0, 5))))

## 3. Pick a temperature per task

Fill the right column with low (~0) or high (~0.7–1.0) and one reason each:

| Task | Temperature | Why |
| --- | --- | --- |
| Classify a support ticket |  |  |
| Brainstorm product names |  |  |
| Extract a date into JSON |  |  |
| Route to a tool (L07) |  |  |

## 4. Why temperature 0 still varies

In 2–3 sentences: why can `temperature=0` still give different answers across runs?

_Your answer:_ 

[↑ Back to top](#top)